<a href="https://colab.research.google.com/github/tousif47/Geostormcaster/blob/main/omni_parse_1m_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
"""
omni_parse_1m.py — OMNI high-resolution 1-minute .asc files -> parquet
Source: https://spdf.gsfc.nasa.gov/pub/data/omni/high_res_omni/
        (omni_minYYYY.asc, coverage 1995+ used; pre-1995 plasma is sparse)
Format: verified against hroformat.txt in the same directory.

Output: omni_1m_clean.parquet — 1-minute solar wind, 1995 to present.
        No Kp here (1-min files do not carry it); labels join from the
        hourly omni_clean.csv in notebook 04.
Self-check at the end validates coverage ramp-up and physical ranges.
"""
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/temp")   # <- folder with the .asc files
OUT      = "/content/drive/MyDrive/temp/omni_1m_clean.parquet"
START_YEAR = 1995                                    # plasma coverage is poor before this

# full 46-field HRO layout (hroformat.txt); only the kept columns survive
COLS = [
    'year', 'doy', 'hour', 'minute',
    'imf_sc', 'sw_sc', 'n_imf', 'n_plasma', 'pct_interp',
    'timeshift', 'rms_timeshift', 'rms_pfn', 'dbot',
    'b_magnitude', 'bx', 'by_gse', 'bz_gse', 'by_gsm', 'bz_gsm',
    'rms_b_scalar', 'rms_b_vector',
    'speed', 'vx', 'vy', 'vz', 'density', 'temp',
    'flow_pressure', 'electric_field', 'plasma_beta', 'alfven_mach',
    'sc_x', 'sc_y', 'sc_z', 'bsn_x', 'bsn_y', 'bsn_z',
    'ae', 'al', 'au', 'sym_d', 'sym_h', 'asy_d', 'asy_h',
    'pcn', 'ms_mach',
]

KEEP = ['timestamp', 'b_magnitude', 'by_gsm', 'bz_gsm', 'speed', 'density', 'temp']

# HRO fill values by field format (F8.2 -> 9999.99, F8.1 -> 99999.9,
# F7.2 -> 999.99, F9.0 -> 9999999.0)
NA_VALUES = [9999.99, 99999.9, 999.99, 9999999.0, 999.9, 999999.0]

frames = []
for path in sorted(DATA_DIR.glob("omni_min*.asc")):
    year = int(path.stem[-4:])
    if year < START_YEAR:
        print(f"skip {path.name} (pre-{START_YEAR})")
        continue
    df = pd.read_csv(path, delim_whitespace=True, names=COLS,
                     na_values=NA_VALUES, low_memory=False)
    df['timestamp'] = pd.to_datetime(df['year'], format='%Y') \
        + pd.to_timedelta(df['doy'] - 1, unit='D') \
        + pd.to_timedelta(df['hour'], unit='h') \
        + pd.to_timedelta(df['minute'], unit='m')
    frames.append(df[KEEP])
    print(f"{path.name}: {len(df)} rows")

df = pd.concat(frames, ignore_index=True).set_index('timestamp').sort_index()

# a complete year has 525600 minutes (527040 in leap years); last year may be partial
full_years = df[df.index.year < df.index.year.max()]
assert full_years.groupby(full_years.index.year).size().ge(525600).all(), \
    "bad row count — check .asc files"
df = df[~df.index.duplicated(keep='first')]

df = df.astype(np.float32)

# --- self-check ---
# coverage: valid plasma fraction should ramp up through 1995-1998 toward ~1.0
cov = df['speed'].notna().groupby(df.index.year).mean()
print("\n===== SELF-CHECK =====")
print("plasma coverage by year (first 8 / last 3):")
print(pd.concat([cov.head(8), cov.tail(3)]).round(3).to_string())
for c in KEEP[1:]:
    s = df[c].dropna()
    print(f"{c:12s} n={len(s):9d}  min={s.min():10.2f}  "
          f"max={s.max():10.2f}  mean={s.mean():10.2f}")

df.to_parquet(OUT)
print(f"\nSaved {OUT}: {len(df)} rows")

/tmp/ipykernel_744/1057648489.py:46: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(path, delim_whitespace=True, names=COLS,


omni_min1995.asc: 525600 rows


/tmp/ipykernel_744/1057648489.py:46: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(path, delim_whitespace=True, names=COLS,


omni_min1996.asc: 527040 rows


/tmp/ipykernel_744/1057648489.py:46: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(path, delim_whitespace=True, names=COLS,


omni_min1997.asc: 525600 rows


... (54 outputs left out)

/tmp/ipykernel_744/1057648489.py:46: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(path, delim_whitespace=True, names=COLS,


omni_min2025.asc: 525600 rows


/tmp/ipykernel_744/1057648489.py:46: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(path, delim_whitespace=True, names=COLS,


omni_min2026.asc: 525600 rows

===== SELF-CHECK =====
plasma coverage by year (first 8 / last 3):
timestamp
1995    0.825
1996    0.832
1997    0.844
1998    0.849
1999    0.856
2000    0.862
2001    0.840
2002    0.848
2024    0.688
2025    0.749
2026    0.561
b_magnitude  n= 15549310  min=      0.12  max=     72.91  mean=      5.88
by_gsm       n= 15549310  min=    -64.79  max=     69.90  mean=     -0.01
bz_gsm       n= 15549310  min=    -60.59  max=     69.70  mean=     -0.06
speed        n= 13286397  min=    180.70  max=   1137.40  mean=    429.39
density      n= 13286404  min=      0.03  max=     80.95  mean=      6.44
temp         n= 13277603  min=    543.00  max=7617765.00  mean=  93473.35

Saved /content/drive/MyDrive/temp/omni_1m_clean.parquet: 16830720 rows
